# ViT Paper Replication

Replicating *An Image is Worth 16x16 Words* (Dosovitskiy et al., 2020) on FoodVision Mini (pizza / steak / sushi).

## 1. Setup

In [ ]:
# Uncomment to install in Colab
# !pip install torch torchvision torchinfo matplotlib Pillow tqdm requests

import torch
from torch import nn
from torchvision import transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import random

from vit.data import download_data, create_dataloaders
from vit.model import (
    PatchEmbedding, MultiHeadSelfAttentionBlock, MLPBlock,
    TransformerEncoderBlock, ViT,
)
from vit.engine import train
from vit.utils import save_model, load_model, plot_loss_curves, pred_and_plot_image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
torch.manual_seed(42)

## 2. Data Exploration

In [ ]:
DATA_URL = 'https://github.com/mrdbourke/pytorch-deep-learning/releases/download/misc/pizza_steak_sushi.zip'
image_path = download_data(source=DATA_URL, destination='pizza_steak_sushi')
print(f'Data path: {image_path}')

In [ ]:
# Sample image grid
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for i, class_name in enumerate(['pizza', 'steak', 'sushi']):
    class_dir = image_path / 'train' / class_name
    images = list(class_dir.glob('*.jpg'))[:4]
    for j, img_path in enumerate(images):
        axes[i, j].imshow(mpimg.imread(img_path))
        axes[i, j].set_title(class_name, fontsize=9)
        axes[i, j].axis('off')
plt.suptitle('FoodVision Mini — Sample Images', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Class distribution
class_names = ['pizza', 'steak', 'sushi']
counts = [len(list((image_path / 'train' / c).glob('*.jpg'))) for c in class_names]
plt.bar(class_names, counts, color=['tomato', 'steelblue', 'gold'])
plt.title('Training samples per class')
plt.ylabel('Count')
plt.show()

## 3. Patch Embedding Walkthrough

How a 224×224 image becomes 196 patch tokens.

In [ ]:
img = torch.randn(1, 3, 224, 224)
print(f'Input:            {img.shape}')  # [1, 3, 224, 224]

patcher = PatchEmbedding()
patches = patcher(img)
print(f'After PatchEmbed: {patches.shape}')  # [1, 196, 768]

class_token = nn.Parameter(torch.randn(1, 1, 768))
x = torch.cat([class_token.expand(1, -1, -1), patches], dim=1)
print(f'+ class token:    {x.shape}')  # [1, 197, 768]

pos_embed = nn.Parameter(torch.randn(1, 197, 768))
x = x + pos_embed
print(f'+ pos embedding:  {x.shape}')  # [1, 197, 768]

In [ ]:
# Visualise the 196 patches extracted from a sample image
sample_img_path = next((image_path / 'train' / 'pizza').glob('*.jpg'))
img_tensor = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224), antialias=True),
])(plt.imread(sample_img_path))

fig, axes = plt.subplots(14, 14, figsize=(8, 8))
patch_imgs = img_tensor.unfold(1, 16, 16).unfold(2, 16, 16)  # [3, 14, 14, 16, 16]
for row in range(14):
    for col in range(14):
        patch = patch_imgs[:, row, col, :, :].permute(1, 2, 0).numpy()
        axes[row, col].imshow(patch.clip(0, 1))
        axes[row, col].axis('off')
plt.suptitle('196 patches (16×16 each)', fontsize=12)
plt.tight_layout()
plt.show()

## 4. ViT Architecture from Scratch

In [ ]:
try:
    from torchinfo import summary
    print('--- PatchEmbedding ---')
    summary(PatchEmbedding(), input_size=(1, 3, 224, 224),
            col_names=['input_size', 'output_size', 'num_params'])
except ImportError:
    print('pip install torchinfo for layer summaries')

In [ ]:
try:
    from torchinfo import summary
    print('--- MultiHeadSelfAttentionBlock ---')
    summary(MultiHeadSelfAttentionBlock(), input_size=(1, 197, 768))
except ImportError:
    pass

In [ ]:
try:
    from torchinfo import summary
    print('--- Full ViT-Base/16 ---')
    summary(ViT(), input_size=(1, 3, 224, 224))
except ImportError:
    pass

## 5. Training from Scratch

In [ ]:
custom_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
train_dl, test_dl, class_names = create_dataloaders(
    train_dir=image_path / 'train',
    test_dir=image_path / 'test',
    transform=custom_transform,
    batch_size=32,
)

In [ ]:
custom_vit = ViT(num_classes=len(class_names)).to(device)
optimizer = torch.optim.Adam(custom_vit.parameters(), lr=3e-3, weight_decay=0.1)
loss_fn = nn.CrossEntropyLoss()

custom_results = train(
    model=custom_vit,
    train_dataloader=train_dl,
    test_dataloader=test_dl,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=10,
    device=device,
)

In [ ]:
plot_loss_curves(custom_results, show=True)

## 6. Pretrained ViT-B/16 Fine-tuning

In [ ]:
weights = ViT_B_16_Weights.DEFAULT
pretrained_vit = vit_b_16(weights=weights)
for param in pretrained_vit.parameters():
    param.requires_grad = False
pretrained_vit.heads = nn.Linear(in_features=768, out_features=len(class_names))
pretrained_vit = pretrained_vit.to(device)
print(f'Trainable params: {sum(p.numel() for p in pretrained_vit.parameters() if p.requires_grad):,}')

In [ ]:
auto_transforms = weights.transforms()
train_dl_pt, test_dl_pt, _ = create_dataloaders(
    train_dir=image_path / 'train',
    test_dir=image_path / 'test',
    transform=auto_transforms,
    batch_size=32,
)
optimizer_pt = torch.optim.Adam(pretrained_vit.parameters(), lr=1e-3)
pretrained_results = train(
    model=pretrained_vit,
    train_dataloader=train_dl_pt,
    test_dataloader=test_dl_pt,
    optimizer=optimizer_pt,
    loss_fn=loss_fn,
    epochs=5,
    device=device,
)

In [ ]:
plot_loss_curves(pretrained_results, show=True)

## 7. Results Comparison

In [ ]:
header = f"{'Model':<30} {'Test Acc':>10} {'Test Loss':>12}"
print(header)
print('-' * 55)
print(f"{'ViT from scratch (10 ep)':<30} "
      f"{custom_results['test_acc'][-1]:>10.3f} "
      f"{custom_results['test_loss'][-1]:>12.4f}")
print(f"{'Pretrained ViT-B/16 (5 ep)':<30} "
      f"{pretrained_results['test_acc'][-1]:>10.3f} "
      f"{pretrained_results['test_loss'][-1]:>12.4f}")

## 8. Predictions on Test Images

In [ ]:
test_images = (
    list((image_path / 'test' / 'pizza').glob('*.jpg'))[:2] +
    list((image_path / 'test' / 'steak').glob('*.jpg'))[:2] +
    list((image_path / 'test' / 'sushi').glob('*.jpg'))[:2]
)
random.shuffle(test_images)

for img_path in test_images:
    pred_and_plot_image(
        model=custom_vit,
        image_path=str(img_path),
        class_names=class_names,
        device=device,
        transform=custom_transform,
    )